<a href="https://colab.research.google.com/github/ttktjmt/g1_spinkick_example/blob/ttktjmt%2Ftrain-with-colab/notebooks/g1_spinkick_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🤖 G1 Spinkick Example Tutorial with MJLab**

### **📦 Setup and Installation**

In [ ]:
# Clone the g1_spinkick_example repository
!if [ ! -d "g1_spinkick_example" ]; then git clone -q https://github.com/ttktjmt/g1_spinkick_example.git; fi
%cd /content/g1_spinkick_example

!uv sync

print("✓ Installation complete!")

### **🔑 WandB Login**

In [ ]:
!wandb login

### **⬇️ Download assets and motion data**

In [ ]:
SHAREPOINT_URL = "https://1sfu-my.sharepoint.com/:u:/g/personal/xbpeng_sfu_ca/EclKq9pwdOBAl-17SogfMW0Bved4sodZBQ_5eZCiz9O--w?e=bqXBaa&download=1"
OUT = "mimickit.zip"

!wget -O "$OUT" "$SHAREPOINT_URL"

In [ ]:
import os

# Create the data directory if it doesn't exist
DATA_DIR = "/content/g1_spinkick_example/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Unzip the archive into the data directory
!unzip -o {OUT} -d {DATA_DIR}

print(f"✓ Archive '{OUT}' extracted to '{DATA_DIR}/' successfully!")

### **🔄 Convert**

Create a custom registry named "Motions" as described in the [mjlab README](https://github.com/mujocolab/mjlab?tab=readme-ov-file#2-motion-imitation)

Convert pkl to csv

In [ ]:
!uv run pkl_to_csv.py \
    --pkl-file /content/g1_spinkick_example/data/MimicKit_Data/motions/g1/g1_spinkick.pkl \
    --csv-file g1_spinkick.csv \
    --duration 2.65 \
    --add-start-transition \
    --add-end-transition \
    --transition-duration 0.5 \
    --pad-duration 1.0

Convert csv to npz

In [ ]:
!MUJOCO_GL=egl uv run -m mjlab.scripts.csv_to_npz \
    --input-file g1_spinkick.csv \
    --output-name mimickit_spinkick_safe \
    --input-fps 60 \
    --output-fps 50 \
    --render True

## **🏋️ Train**

In [ ]:
!MUJOCO_GL=egl CUDA_VISIBLE_DEVICES=0 uv run train \
    Mjlab-Spinkick-Unitree-G1 \
    --registry-name ${WANDB_ENTITY}/csv_to_npz/mimickit_spinkick_safe \
    --env.scene.num-envs 4096 \
    --agent.max-iterations 2_001 \
    --agent.save-interval 400

### **🌐 Launch Viser API**

In [ ]:
import subprocess
import sys
import re

wandb_entity = os.environ["WANDB_ENTITY"]
run_id = "6fo2umjl"  # @param {type:"string"}

process = subprocess.Popen(
  [
    "uv",
    "run",
    "play",
    "Mjlab-Spinkick-Unitree-G1",
    "--wandb-run-path",
    f"{wandb_entity}/mjlab/{run_id}",
    "--num_envs",
    "8",
  ],
  stdout=subprocess.PIPE,
  stderr=subprocess.STDOUT,
  universal_newlines=True,
  bufsize=1,
)

detected_port = None

for line in process.stdout:
  print(line, end="")
  sys.stdout.flush()

  # Extract port number from viser output
  port_match = re.search(r":(\d{4})", line)
  if port_match and "viser" in line.lower():
    detected_port = int(port_match.group(1))
    print("\n" + "=" * 34)
    print(f"✅ Server is running on port {detected_port}!")
    print("=" * 34)
    break

### **🖥️ Embed Client as iframe**

In [ ]:
from google.colab import output

port = detected_port if detected_port else 8080
output.serve_kernel_port_as_iframe(port=port, height=600)